# MVR constrained HMMs: an overview

An **MVR** (mediation variable representation) is a deterministic automaton over
hidden-state sequences. It carries a *mediation* state alongside the HMM's hidden
state — `ini` starts it, `upd` steps it, `evl` accepts — and a hidden path is
feasible exactly when the automaton accepts it. Every algorithm below runs over the
**augmented** chain, hidden × mediation.

This notebook provides an overview: how a constraint is built, and what each of the seven
algorithms does with it. The per-algorithm notebooks in this folder go deeper on one
each; this one shows them side by side on a single model so the interfaces can be read
against each other.

| Query | Function | Answers |
| --- | --- | --- |
| Constrained MAP | `viterbi_torch_mvr_chmm` | the most likely feasible hidden path |
| Marginal MAP | `marginal_map_torch_mvr_chmm` | the most likely assignment at a few chosen times, summing out the rest |
| Satisfaction probability | `sat_prob_torch_mvr_chmm` | how likely a constraint is to hold |
| Satisfaction time | `sat_time_torch_mvr_chmm` | *when* it first holds |
| Posterior sampling | `ffbs_torch_mvr_chmm` | feasible paths drawn from the posterior |
| Stopped sampling | `stopped_sampling_torch_mvr_chmm` | feasible paths that stop when a constraint first holds |
| Constrained smoothing | `forward_backward_mvr_chmm` | the posterior over hidden states, given the observations *and* feasibility |
| Constrained learning | `baum_welch_mvr_chmm` | an EM fit of the HMM under that posterior |

They all take the same object — an `MVR_CHMM` — so the only thing that changes between
the calls in this notebook is the constraint.

NOTE: None of these are exported from `conin`, since `torch` is not a declared dependency;
import them by full path, as below.

In [ ]:
import itertools
import warnings

import numpy as np
import matplotlib.pyplot as plt
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.mvr_constraints import (
    mvr_current_state,
    mvr_forbid_state,
)
from conin.hidden_markov_model.mvr_operators import (
    mvr_already_satisfied,
    mvr_count,
    mvr_not_yet,
    mvr_precedence,
)
from conin.hidden_markov_model.mvr_formula import build_mvr

from conin.hidden_markov_model.inference.viterbi_mvr import viterbi_torch_mvr_chmm
from conin.hidden_markov_model.inference.marginal_map_mvr import (
    marginal_map_torch_mvr_chmm,
)
from conin.hidden_markov_model.other_queries.sat_prob_mvr import (
    sat_prob_torch_mvr_chmm,
)
from conin.hidden_markov_model.other_queries.sat_time_mvr import (
    sat_time_torch_mvr_chmm,
)
from conin.hidden_markov_model.sampling.ffbs_mvr import ffbs_torch_mvr_chmm
from conin.hidden_markov_model.sampling.stopped_sampling_mvr import (
    stopped_sampling_torch_mvr_chmm,
)
from conin.hidden_markov_model.learning.baum_welch_mvr import (
    baum_welch_mvr_chmm,
    forward_backward_mvr_chmm,
)


HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={
        "A": 0.28,
        "B": 0.40,
        "C": 0.32,
    },
    transition_probs={
        ("A", "A"): 0.34,
        ("A", "B"): 0.05,
        ("A", "C"): 0.61,
        ("B", "A"): 0.48,
        ("B", "B"): 0.06,
        ("B", "C"): 0.46,
        ("C", "A"): 0.43,
        ("C", "B"): 0.18,
        ("C", "C"): 0.39,
    },
    emission_probs={
        ("A", "lo"): 0.20,
        ("A", "mid"): 0.31,
        ("A", "hi"): 0.49,
        ("B", "lo"): 0.54,
        ("B", "mid"): 0.23,
        ("B", "hi"): 0.23,
        ("C", "lo"): 0.09,
        ("C", "mid"): 0.01,
        ("C", "hi"): 0.90,
    },
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)


def run(mvr, path):
    """Evaluate one MVR on a hidden path, by hand."""
    state = mvr.ini[path[0]]

    for h in path[1:]:
        state = mvr.upd[(state, h)]

    return mvr.evl[state]


def holds(mvrs, path):
    """A constraint set holds iff every MVR in it holds."""
    return all(run(mvr, path) for mvr in mvrs)


def model(*formulas, base=hmm):
    """One MVR_CHMM enforcing every formula given."""
    return MVR_CHMM(
        hidden_markov_model=base,
        constraints=[mvr for f in formulas for mvr in build_mvr(base, f)],
    )


print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## 1. Three ways to write the same constraint

Consider the constraint: *never visit A*. It can be constructed directly, using a composition of
constructors and operators, or from a formula string. All three approaches give the same `mvr` object.

1. Specify a **`HomMVR` directly.**  Construct the MVR by hand by defining the maps `ini`, `upd`, `evl` and the mediation space.
2. **`mvr_constraints` + `mvr_operators`.** Constructors build atomic classes, operators
   combine them. `mvr_forbid_state` can be defined as `mvr_not_yet(mvr_current_state)`.
3. **`mvr_formula`.** One can pass a structured formula into `build_mvr`. Alternatively, passing this formula into `build_mvr_functor` creates a factory for that constraint.

Option 3 is the user-friendly default. Option 2 can be used when the grammar cannot specify the constraint. Option 1 when the constraint can't be built from the library's constructors and operators. See [`formula.ipynb`](formula.ipynb) and
[`FORMULA_CHEATSHEET.md`](FORMULA_CHEATSHEET.md) for the syntax.

NOTE: While all three options generate an `mvr` for the same constraint, the sizes can differ:
the composed forms are products of their parts and are NOT minimized afterwards.

In [ ]:
mediation_states = ["ok", "violated"]

by_hand = HomMVR(
    hidden_states=HIDDEN_STATES,
    mediation_states=mediation_states,
    ini={h: ("violated" if h == "A" else "ok") for h in HIDDEN_STATES},
    upd={
        (m, h): ("violated" if m == "violated" or h == "A" else "ok")
        for m in mediation_states
        for h in HIDDEN_STATES
    },
    evl={"ok": True, "violated": False},
)

routes = {
    "HomMVR, by hand": by_hand,
    "mvr_forbid_state(hmm, {'A'})": mvr_forbid_state(hmm, {"A"}),
    "mvr_not_yet(mvr_current_state(...))": mvr_not_yet(mvr_current_state(hmm, {"A"})),
    'build_mvr(hmm, "never A")': build_mvr(hmm, "never A")[0],
}

paths = [list(p) for p in itertools.product(HIDDEN_STATES, repeat=5)]
reference = [run(by_hand, path) for path in paths]

for label, mvr in routes.items():
    same = [run(mvr, path) for path in paths] == reference
    print(f"{label:<38} {len(mvr.mediation_states)} mediation states   agrees: {same}")

print(f"\nchecked on all {len(paths)} hidden paths of length 5")

## 2. Simple and compound constraints

A **simple** constraint is one atom under one operator. A **compound** one nests
operators. We consider one example of each type and their formulas:
- `never A`: never hit state `A.
- `reach B before A[2]`: visit `B` before the second visit to `A`.

When specifying constraints from formulas, keep in mind:

1. A top-level `and` **splits into a list** of constraints rather than building the
  product automaton, which is cheaper for downstream algorithms. An `and` nested
  under another operator still builds the product.
2. `between a and b` attaches a **`time_range`**, the window over which the constraint is
  enforced. The automaton is initialized at the start of that window and evaluated at its
  end; outside it, the constraint contributes nothing at all.


In [ ]:
SIMPLE = "never A"
COMPOUND = "reach B before A[2]"

for formula in [
    SIMPLE,
    "reach B",
    "count(B) in [2,3]",
    COMPOUND,
    "count(A) <= 1 and reach B",
    "(reach B and never C) between 0 and 3",
]:
    built = build_mvr(hmm, formula)
    print(
        f"{formula:<40} {len(built)} constraint(s), "
        f"{[len(mvr.mediation_states) for mvr in built]} mediation states, "
        f"windows {[mvr.time_range for mvr in built]}"
    )

# The compound constraint, lowered by hand rather than parsed.
by_ops = mvr_precedence(
    [
        mvr_already_satisfied(mvr_current_state(hmm, {"B"})),
        mvr_count(mvr_current_state(hmm, {"A"}), "2"),
    ],
    "<",
)
by_formula = build_mvr(hmm, COMPOUND)[0]

agree = all(run(by_ops, p) == run(by_formula, p) for p in paths)
print(f'\n"{COMPOUND}" matches its hand-lowered form on all {len(paths)} paths: {agree}')

## 3. Inference —  Constrained MAP and Marginal-MAP

For constrained MAP, `viterbi_torch_mvr_chmm` decodes the highest-scoring hidden path that satisfies every
constraint. `marginal_map_torch_mvr_chmm` performs constrained marginal-MAP: it maximizes over a
chosen set of `query_times` and **sums out** every other time.

In this example, `never A` changes the MAP path while `reach B before A[2]` does not, as the
unconstrained path already satisfies it. That does not mean the constraint is inert. The sampling section shows it affecting the posterior.

- For each constraint, we carry perform constrained MAP over the full latent path. This is the "viterbi" column.
- The "mMAP" column shows the constrained marginal-MAP over times `0,3,6`. The "viterbi there" column for comparison shows the restriction of the MAP full path to those times. This is to emphasize that, in general, Viterbi + restriction does not give the same answers as a marginal-MAP.

In [ ]:
QUERY = [0, 3, 6]

header = f"{'formula':<24} {'viterbi':<16} {'log P':>9}   {'mMAP at 0,3,6':<14} {'viterbi there':<14}"
print(header)
print("-" * len(header))

for formula in [None, SIMPLE, COMPOUND]:
    chmm = model(*([] if formula is None else [formula]))

    path, score = viterbi_torch_mvr_chmm(
        chmm, observed, return_augmented=False, return_score=True
    )
    mmap = marginal_map_torch_mvr_chmm(
        chmm, observed, query_times=QUERY, return_augmented=False
    )

    print(
        f"{formula or '(unconstrained)':<24} {' '.join(path):<16} {score:9.4f}   "
        f"{' '.join(mmap):<14} {' '.join(path[t] for t in QUERY):<14}"
    )

## 4. Constrained Sampling

There are two algorithms for constrained sampling:

1. `ffbs_torch_mvr_chmm` does constrained sampling for a fixed time horizon $T$. It draws paths from:
    $$P(x_{1:T} | \text{observations, constraints satisfied})$$
3. `stopped_sampling_torch_mvr_chmm` does constrained stopped sampling for a regular stopping time $\tau$: It draws paths from:
    $$P(x_{1:\tau} | \text{observations, constraints satisfied})$$

We first look at the effect of constraints on the sampled paths by counting how often each state is occupied at each time shows what the constraint actually
does to the posterior, comparing against the unconstrained model. `never A` pins `A` to exactly zero and moves its mass onto `B` and `C`.
`reach B before A[2]` never mentions `C` and barely moves it, but pulls `B` earlier and
pushes `A` later — `A` at `t=0` drops from 0.20 to 0.05 — which is the constraint doing
precisely what it says.

In [ ]:
def occupancy(*formulas, num_samples=20_000, seed=0):
    """P(hidden = h at time t), estimated from feasible FFBS draws."""
    draws = ffbs_torch_mvr_chmm(
        model(*formulas),
        observed,
        num_samples=num_samples,
        generator=torch.Generator().manual_seed(seed),
    )

    return {
        h: np.array([np.mean([path[t] == h for path in draws]) for t in range(T)])
        for h in HIDDEN_STATES
    }


curves = [
    ("unconstrained", occupancy(), "0.6", "o--"),
    (SIMPLE, occupancy(SIMPLE), "C3", "s-"),
    (COMPOUND, occupancy(COMPOUND), "C0", "o-"),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)

for ax, h in zip(axes, HIDDEN_STATES):
    for label, occ, color, style in curves:
        ax.plot(range(T), occ[h], style, color=color, label=label, ms=4)
    ax.set_title(f"P(hidden = {h})")
    ax.set_xlabel("t")
    ax.set_ylim(-0.03, 1.0)

axes[0].set_ylabel("occupancy")
axes[0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

for label, occ, _, _ in curves:
    print(f"{label:<22} t=0: " + "   ".join(f"{h}={occ[h][0]:.3f}" for h in HIDDEN_STATES))

Next, we showcase stopped sampling. Now, our paths stop at the first `B`, a random condition. Our paths also incorporate the previous two constraints. Forbidding `A` concentrates the stopping time at `t = 1` — with
`A` gone and the first observation favouring `C`, `B` arrives almost immediately.

In [ ]:
for enforced in [(), (SIMPLE,)]:
    paths, taus = stopped_sampling_torch_mvr_chmm(
        model(*enforced, "reach B"),
        observed,
        target="reach B",
        num_samples=4_000,
        generator=torch.Generator().manual_seed(1),
        return_times=True,
    )

    share = np.bincount(taus.numpy(), minlength=T) / len(paths)
    label = ", ".join(enforced) or "nothing"

    print(f"stopping time, enforcing {label:<10} "
          + "  ".join(f"{t}:{s:.3f}" for t, s in enumerate(share)))
    print("   sampled prefixes: " + " | ".join(" ".join(p) for p in paths[:4]))

## 5. Constrained Learning

`forward_backward_mvr_chmm` returns the constrained posterior — `gamma` over hidden
states, `xi` over consecutive pairs, and `log P(y, constraints satisfied | theta)`.
`baum_welch_mvr_chmm` runs EM on top of it.

The generative model is the **unconstrained** HMM; feasibility is an observed fact about
each realized hidden path, so the indicator carries no parameters and the M-step stays the
ordinary weighted-count normalization. That makes this exact, closed-form, **monotone** EM
with no partition-function term.

The fit beats the *true* model on this objective, which is not a bug: the objective is
`log P(y, constraint | theta)` on constraint-filtered data, and the true model is not its
maximizer. [`learning_BW.ipynb`](learning_BW.ipynb) works that through.

In [ ]:
rng = np.random.default_rng(0)
repn = hmm.repn


def sample_feasible(formula, n):
    """Forward-sample from the HMM, keeping the paths the formula accepts."""
    mvrs, out = build_mvr(hmm, formula), []

    while len(out) < n:
        hidden = [HIDDEN_STATES[rng.choice(3, p=repn.start_vec)]]
        for _ in range(T - 1):
            row = repn.transition_mat[hmm.hidden_to_internal[hidden[-1]]]
            hidden.append(HIDDEN_STATES[rng.choice(3, p=row)])

        if holds(mvrs, hidden):
            out.append([
                OBSERVED_STATES[rng.choice(3, p=repn.emission_mat[hmm.hidden_to_internal[h]])]
                for h in hidden
            ])

    return out


def flat_hmm():
    """An uninformative starting point, with no structural zeros."""
    rows = [[0.5, 0.3, 0.2], [0.2, 0.5, 0.3], [0.3, 0.2, 0.5]]
    start = HiddenMarkovModel()
    start.load_model(
        start_probs={h: 1 / 3 for h in HIDDEN_STATES},
        transition_probs={
            (a, b): rows[i][j]
            for i, a in enumerate(HIDDEN_STATES)
            for j, b in enumerate(HIDDEN_STATES)
        },
        emission_probs={
            (h, o): rows[i][j]
            for i, h in enumerate(HIDDEN_STATES)
            for j, o in enumerate(OBSERVED_STATES)
        },
        initialize=True,
    )
    return start


for formula in [SIMPLE, COMPOUND]:
    sequences = sample_feasible(formula, 60)

    gamma, xi, loglik = forward_backward_mvr_chmm(model(formula), sequences[0])
    fitted, history = baum_welch_mvr_chmm(
        model(formula, base=flat_hmm()), sequences, max_iter=60, tol=0
    )

    def objective(candidate):
        """Total log P(y, constraint satisfied | theta) over the data set."""
        return sum(
            forward_backward_mvr_chmm(model(formula, base=candidate), s)[2]
            for s in sequences
        )

    print(f"{formula}")
    print(f"   forward-backward  loglik {loglik:8.4f}   gamma {tuple(gamma.shape)}   xi {tuple(xi.shape)}")
    print(f"   EM over {len(sequences)} sequences   {history[0]:9.3f} -> {history[-1]:9.3f}"
          f"   monotone {bool((np.diff(history) >= -1e-9).all())}")
    print(f"   objective    flat {objective(flat_hmm()):9.3f} |"
          f" fitted {objective(fitted):9.3f} | true {objective(hmm):9.3f}\n")

## 6. Other Constraint Queries: Satisfaction Probability and Satisfaction Time Distribution

These two queries are about a constraint's properties. Both single out a constraint as the
**target**, chosen by index or by name. Every other constraint in the model is enforced
as usual.

- `sat_prob_torch_mvr_chmm` — the probability the target holds, evaluated at the end of
  its window. This is NOT "holds at some point": for that, target `mvr_already_satisfied(...)`,
  whose accept state is absorbing. It computes:
   $$ P(\text{target is satisfied} \ \mid \ \text{other constraints}, \ \text{observations}) $$
- `sat_time_torch_mvr_chmm` — the distribution of the **first** satisfaction time of the target, over its
  window. It computes the distribution:
  $$ P(\text{target first satisfied at time = }t \ \mid  \text{other constraints}, \ \text{observations}) $$

Enforcing another constraint alongside the target can change the answer, which we show in the first block: forbidding `A` outright makes *B before the second A* nearly free.
It also changes *when* the target first holds, which the figure below shows for the same three cases.

In [ ]:
for enforced in [(), (SIMPLE,), ("count(A) <= 1",)]:
    # target= takes the conjunct's own text, since that is what build_mvr names it.
    prob = sat_prob_torch_mvr_chmm(
        model(*enforced, COMPOUND), observed, target=COMPOUND
    )
    also = ", ".join(enforced) or "nothing"
    print(f"P(satisfied | y, also enforcing {also}) = {prob:.4f}")


In [ ]:
# The same three cases, now as first-satisfaction-time distributions.
series = []

for enforced, color in [((), "0.55"), ((SIMPLE,), "C3"), (("count(A) <= 1",), "C0")]:
    times, probs = sat_time_torch_mvr_chmm(
        model(*enforced, COMPOUND), observed, target=COMPOUND
    )
    series.append((", ".join(enforced) or "nothing else enforced", times, probs.numpy(), color))

fig, ax = plt.subplots(figsize=(8, 3.4))
width = 0.26

for i, (label, times, probs, color) in enumerate(series):
    ax.bar(
        np.array(times) + (i - 1) * width,
        probs,
        width * 0.92,
        label=label,
        color=color,
        zorder=3,
    )

ax.set_xticks(series[0][1])
ax.set_xlabel("t   (first time the target holds)")
ax.set_ylabel("probability")
ax.set_title(f'When does "{COMPOUND}" first hold?', fontsize=11)
ax.grid(axis="y", color="0.9", lw=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

for label, times, probs, _ in series:
    print(f"{label:<22} mode t={times[int(probs.argmax())]}   "
          + "  ".join(f"{t}:{p:.3f}" for t, p in zip(times, probs)))


Both are exact, so they can be checked against enumeration — 3⁷ = 2187 paths here.

In [ ]:
def path_weight(path):
    """P(hidden path, observations)."""
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = np.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += np.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in enumerate(observed):
        total += np.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return np.exp(total)


constraints = build_mvr(hmm, COMPOUND)
evidence = satisfied = 0.0

for path in itertools.product(HIDDEN_STATES, repeat=T):
    weight = path_weight(list(path))
    evidence += weight
    satisfied += weight * holds(constraints, list(path))

print(f"brute force  P(satisfied | y) = {satisfied / evidence:.6f}")
print(f"sat_prob     P(satisfied | y) = {sat_prob_torch_mvr_chmm(model(COMPOUND), observed, target=COMPOUND):.6f}")

## Where to go next

Every algorithm above took the same `MVR_CHMM`, and the constraint was the only thing that
changed between them. That is the whole design: write the constraint once, then ask any
question of it.

| Notebook | Goes deeper on |
| --- | --- |
| [`formula.ipynb`](formula.ipynb) | the formula layer, and the automaton it replaces |
| [`FORMULA_CHEATSHEET.md`](FORMULA_CHEATSHEET.md) | the full syntax, precedence, and the pitfalls |
| [`regex_tutorial.ipynb`](regex_tutorial.ipynb) | `mvr_regex`, and the deferred `ConstrainedHiddenMarkovModel` functor path |
| [`viterbi.ipynb`](viterbi.ipynb) | windowed constraints, user-defined horizons, intermittent observations |
| [`marginal_map.ipynb`](marginal_map.ipynb) | why marginal MAP disagrees with a restricted Viterbi path |
| [`sat_time.ipynb`](sat_time.ipynb) | first satisfaction vs per-time satisfaction; windowing vs slicing |
| [`sat_prob.ipynb`](sat_prob.ipynb) | satisfied-at-`b` vs satisfied-ever, and a `time_range` sweep |
| [`sampling.ipynb`](sampling.ipynb) | FFBS and stopped sampling against a rejection baseline |
| [`learning_BW.ipynb`](learning_BW.ipynb) | forward-backward against enumeration, and monotone EM |